# 🔀 Routing으로 AI 시스템 효율 끌어올리기

LangGraph 공식 문서의 **Routing 패턴**을 구현합니다. `with_structured_output`을 사용하여 요청을 분류하고 적절한 처리 경로로 라우팅합니다.

> 📢 **Routing 패턴**
> - 사용자 요청을 분석하여 카테고리 분류
> - 구조화된 출력으로 라우팅 로직 구현
> - 각 카테고리별 전문화된 처리
![image](https://mintcdn.com/langchain-5e9cc07a/dL5Sn6Cmy9pwtY0V/oss/images/routing.png?fit=max&auto=format&n=dL5Sn6Cmy9pwtY0V&q=85&s=272e0e9b681b89cd7d35d5c812c50ee6)
### 🎯 학습 목표
1. `with_structured_output`으로 라우팅 스키마 정의
2. `add_conditional_edges`로 동적 분기 구현
3. 다중 전문 에이전트 라우팅

In [ ]:
%pip install -qU langchain langgraph langchain-openai python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

print("✅ 환경 설정 완료!")

✅ 환경 설정 완료!


---
## 1. 라우팅 스키마 정의

`with_structured_output`으로 라우팅 결정을 구조화합니다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 라우팅 스키마 정의
class Route(BaseModel):
    step: Literal["poem", "story", "joke"] = Field(
        description="다음 라우팅 단계 선택"
    )

# LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 구조화된 출력 router
router = llm.with_structured_output(Route)

print("✅ Router 설정 완료!")

---
## 2. State 정의

In [ ]:
from typing import TypedDict

class State(TypedDict):
    input: str      # 사용자 입력
    decision: str   # 라우팅 결정
    output: str     # 최종 출력

print("✅ State 정의 완료!")

---
## 3. 노드 함수 정의

In [ ]:
def llm_call_router(state: State):
    """Router: 입력을 분석하여 적절한 경로 결정"""
    print(f"\n🔀 Router: 입력 분석 중...")
    
    decision = router.invoke([
        SystemMessage(content="사용자 요청을 story, joke, poem 중 하나로 라우팅하세요."),
        HumanMessage(content=state["input"])
    ])
    
    print(f"   📌 선택된 경로: {decision.step}")
    return {"decision": decision.step}

def llm_call_story(state: State):
    """이야기 작성"""
    print(f"\n📖 Story Writer: 이야기 작성 중...")
    result = llm.invoke(f"다음 주제로 짧은 이야기를 써줘: {state['input']}")
    return {"output": result.content}

def llm_call_joke(state: State):
    """농담 작성"""
    print(f"\n😄 Joke Writer: 농담 작성 중...")
    result = llm.invoke(f"다음 주제로 재미있는 농담을 써줘: {state['input']}")
    return {"output": result.content}

def llm_call_poem(state: State):
    """시 작성"""
    print(f"\n📜 Poem Writer: 시 작성 중...")
    result = llm.invoke(f"다음 주제로 아름다운 시를 써줘: {state['input']}")
    return {"output": result.content}

print("✅ 노드 함수 정의 완료!")

---
## 4. 조건부 라우팅 함수

In [ ]:
def route_decision(state: State):
    """라우팅 결정에 따라 다음 노드 선택"""
    if state["decision"] == "story":
        return "llm_call_story"
    elif state["decision"] == "joke":
        return "llm_call_joke"
    elif state["decision"] == "poem":
        return "llm_call_poem"
    else:
        return "llm_call_story"  # 기본값

print("✅ 라우팅 함수 정의 완료!")

---
## 5. 그래프 구성

In [ ]:
from langgraph.graph import StateGraph, START, END

# 워크플로우 구성
builder = StateGraph(State)

# 노드 추가
builder.add_node("llm_call_router", llm_call_router)
builder.add_node("llm_call_story", llm_call_story)
builder.add_node("llm_call_joke", llm_call_joke)
builder.add_node("llm_call_poem", llm_call_poem)

# 엣지 추가
builder.add_edge(START, "llm_call_router")

# 조건부 엣지: 라우터 결정에 따라 분기
builder.add_conditional_edges(
    "llm_call_router",
    route_decision,
    {
        "llm_call_story": "llm_call_story",
        "llm_call_joke": "llm_call_joke",
        "llm_call_poem": "llm_call_poem"
    }
)

# 각 처리 노드에서 END로 연결
builder.add_edge("llm_call_story", END)
builder.add_edge("llm_call_joke", END)
builder.add_edge("llm_call_poem", END)

# 컴파일
router_workflow = builder.compile()
print("✅ Router 워크플로우 컴파일 완료!")

In [ ]:
from IPython.display import Image, display
try:
    display(Image(router_workflow.get_graph().draw_mermaid_png()))
except:
    print("그래프: START → router → [story/joke/poem] → END")

---
## 6. 실행 및 테스트 (스트리밍)

In [ ]:
# 다양한 요청 테스트
test_inputs = [
    "고양이에 대한 농담을 해줘",
    "봄에 관한 시를 써줘",
    "용감한 기사 이야기를 들려줘"
]

for user_input in test_inputs:
    print(f"\n{'='*60}")
    print(f"💬 입력: {user_input}")
    
    for chunk in router_workflow.stream({"input": user_input}, stream_mode="updates"):
        for node_name, updates in chunk.items():
            if "output" in updates:
                print(f"\n🤖 결과:\n{updates['output'][:300]}...")

In [ ]:
# 전체 결과 확인
result = router_workflow.invoke({"input": "인공지능에 대한 재미있는 농담을 해줘"})

print(f"🔀 라우팅 결정: {result['decision']}")
print(f"\n📝 결과:\n{result['output']}")

---
## 💡 정리

### Routing 핵심 개념

| 구성요소 | 역할 | 구현 방법 |
|---------|------|----------|
| **Router** | 요청 분류 | `with_structured_output` |
| **Handler** | 전문 처리 | 개별 노드 |
| **Branching** | 동적 분기 | `add_conditional_edges` |

### 핵심 코드
```python
# 구조화된 라우팅 결정
class Route(BaseModel):
    step: Literal["option_a", "option_b", "option_c"]

router = llm.with_structured_output(Route)

# 조건부 분기
builder.add_conditional_edges(
    "router",
    route_decision,
    {"option_a": "node_a", "option_b": "node_b", "option_c": "node_c"}
)
```

👉 **다음 챕터**: CH03. Langgraph 통한 실제 AI 에이전트 서비스 따라하기